In [3]:
import pandas as pd
import os
import re
from sklearn.preprocessing import StandardScaler
from sklearn import svm
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier


path = "C:/Users/AsusTUF/Downloads/data/"
names = ["accelerometer_X", "accelerometer_Y", "accelerometer_Z", "magnitude"]

name_folder = ["walking", "stairs", "running", "idle"]
rows = []


def fun_magnitude(path):
    df = pd.read_csv(path)
    df["magnitude"] = (
        df["accelerometer_X"] ** 2
        + df["accelerometer_Y"] ** 2
        + df["accelerometer_Z"] ** 2
    ) ** 0.5
    return df


def fun_accelerometer(df, names=names):
    result = []
    for name in names:
        accelerometer_column = df[name]
        average = accelerometer_column.mean()
        swing = accelerometer_column.max() - accelerometer_column.min()
        deviation = accelerometer_column.std()
        result.extend([average, swing, deviation])
    return result


for folder in name_folder:
    p_folder = os.path.join(path, folder)
    files = os.listdir(p_folder)
    for file in files:
        if file.endswith(".csv"):
            p_file = os.path.join(path, folder, file)
            result_df = fun_magnitude(p_file)
            result = fun_accelerometer(result_df)
            result.append(folder)
            number = re.split(r"[.-]", file)
            result.append(int(number[1]))
            rows.append(result)
        else:
            continue

columns_name = [
    "X_mean",
    "X_range",
    "X_std",
    "Y_mean",
    "Y_range",
    "Y_std",
    "Z_mean",
    "Z_range",
    "Z_std",
    "M_mean",
    "M_range",
    "M_std",
    "mark",
    "number",
]
features = pd.DataFrame(rows, columns=columns_name)

print(features.duplicated(subset=["mark", "number"], keep=False).sum())

features.to_csv("D:/G_I/HM/data.csv", index=False)

df = pd.read_csv("D:/G_I/HM/data.csv")

name_features = ["walking", "stairs", "running", "idle"]
study_rows = []
test_rows = []


def sort_sample(name_feature):
    feature = df[df["mark"] == name_feature]
    count = len(feature)
    feature = feature.sort_values("number")

    k = int(count * 80 / 100)

    study = feature.iloc[:k]
    test = feature.iloc[k:]

    return study, test


for name_feature in name_features:
    result = sort_sample(name_feature)
    study_rows.append(result[0])
    test_rows.append(result[1])

df_study = pd.concat(study_rows)
df_test = pd.concat(test_rows)

print(df_study.shape, df_test.shape)
print(df_study["mark"].value_counts())
print(df_test["mark"].value_counts())


X_train_12 = df_study.drop(columns=["mark", "number"])
X_test_12 = df_test.drop(columns=["mark", "number"])

X_train_9 = X_train_12.drop(columns=["M_mean", "M_range", "M_std"])
X_test_9 = X_test_12.drop(columns=["M_mean", "M_range", "M_std"])

Y_train = df_study["mark"]
Y_test = df_test["mark"]


def model(mode, X_train, X_test, count, Y_train=Y_train, Y_test=Y_test):
    dict_result = {}
    if mode == "linear" or mode == "rbf":
        scaler = StandardScaler()

        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)
        clf = svm.SVC(kernel=mode)

    elif mode == 100 or mode == 300:

        clf = RandomForestClassifier(n_estimators=mode, random_state=4)

    clf.fit(X_train, Y_train)

    y_pred = clf.predict(X_test)

    a = accuracy_score(Y_test, y_pred)
    cr = classification_report(Y_test, y_pred, output_dict=True)
    cm = confusion_matrix(Y_test, y_pred)
    print(f"model: {mode} {count}")
    print(a)
    print(classification_report(Y_test, y_pred))
    print(cm)

    dict_result["model"] = mode
    dict_result["features"] = count
    dict_result["macro f1"] = cr["macro avg"]["f1-score"]
    dict_result["f1 stairs"] = cr["stairs"]["f1-score"]
    dict_result["f1 walking"] = cr["walking"]["f1-score"]
    dict_result["accuracy"] = a

    return dict_result


result = []
parameter = [
    ("linear", X_train_9, X_test_9, 9),
    ("linear", X_train_12, X_test_12, 12),
    ("rbf", X_train_9, X_test_9, 9),
    ("rbf", X_train_12, X_test_12, 12),
    (100, X_train_9, X_test_9, 9),
    (100, X_train_12, X_test_12, 12),
    (300, X_train_9, X_test_9, 9),
    (300, X_train_12, X_test_12, 12)
    ]

for m, x_tr, x_te, c in parameter:
    res = model(m, x_tr, x_te, c)
    result.append(res)

table = pd.DataFrame(result)

print(table)

0
(5169, 14) (1293, 14)
mark
running    2726
walking    1480
idle        831
stairs      132
Name: count, dtype: int64
mark
running    682
walking    370
idle       208
stairs      33
Name: count, dtype: int64
model: linear 9
0.9249806651198763
              precision    recall  f1-score   support

        idle       1.00      1.00      1.00       208
     running       1.00      1.00      1.00       682
      stairs       0.11      0.27      0.16        33
     walking       0.93      0.80      0.86       370

    accuracy                           0.92      1293
   macro avg       0.76      0.77      0.75      1293
weighted avg       0.96      0.92      0.94      1293

[[208   0   0   0]
 [  0 682   0   0]
 [  0   0   9  24]
 [  0   0  73 297]]
model: linear 12
0.925754060324826
              precision    recall  f1-score   support

        idle       1.00      1.00      1.00       208
     running       1.00      1.00      1.00       682
      stairs       0.11      0.27      0.16  

# Висновок

